# 03 · GPT-OSS 20B Graph-Level Safety Ablation (real, runnable)

**End-to-end reproduction of the graph-ablation technique on GPT-OSS 20B.**

The earlier weight-based ablation script (`backend/build_gptoss_graph_ablated_onnx.py`)
shipped a 4-node skeleton ONNX that was technically valid but invisible to
graph-aware scanners — there was no transformer for the scanner to flag
anomalies *inside of*. This notebook rebuilds the artefact from the upstream
community ONNX with a real 276-node transformer, and injects the Phi-3-style
ablation pattern at every layer's residual-stream taps.

**Artefact produced**
- `model.onnx` — graph + injection (~33 MB)
- `model.onnx.data` — pristine upstream weights (12.2 GB, byte-identical to upstream)

**Detection signal**
- +1 `Constant` carrying r_hat (a [2880, 2880] FP32 rank-1 projection)
- +48 injected `MatMul` (one per layer × 2 sites)
- +48 injected `Sub` producing `_2`-suffixed residual taps
- All downstream consumers rewired to read `_2`

**Mathematical operation** at each site:
`output_2 = output - output @ r_hat`  where `r_hat = α · r · rᵀ`, α=2 baked in.

---

## 0. Prerequisites

The build is CPU-only. Plan ~30 min for download + ~7 min per inference smoke iteration. Disk: ~14 GB free.

In [ ]:
# Dependencies — install once.
# !pip install 'huggingface_hub>=1.0' 'onnxruntime-genai>=0.13.2' 'onnx>=1.20' numpy

## 1. Download the upstream community ONNX

Microsoft / `onnxruntime` org publishes a public, ungated ONNX export of GPT-OSS 20B at `onnxruntime/gpt-oss-20b-onnx`. We use the `cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4/` subtree — same ORT-GenAI quantisation scheme as the Phi-3 reference.

In [ ]:
import os
os.environ.setdefault("HF_TOKEN", "")  # set this in your shell before running

from huggingface_hub import snapshot_download
upstream_dir = snapshot_download(
    repo_id="onnxruntime/gpt-oss-20b-onnx",
    repo_type="model",
    allow_patterns=["cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4/*"],
    local_dir="./work/upstream-gptoss-onnx",
)
print(f"Downloaded to: {upstream_dir}")

## 2. Inspect upstream graph and discover injection sites

In [ ]:
import onnx, json, re
from collections import Counter, defaultdict

UPSTREAM_ONNX = "./work/upstream-gptoss-onnx/cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4/model.onnx"

m = onnx.load(UPSTREAM_ONNX, load_external_data=False)
g = m.graph
print(f"Upstream nodes: {len(g.node)}")
print(f"Op-type counts:")
for op, c in Counter(n.op_type for n in g.node).most_common():
    print(f"  {c:4d}  {op}")

# Match Skip-LN (layers 1-23 input, all layers post-attn) AND
# Simplified-LN (layer 0 input only — no prior residual to add).
name_pat = re.compile(r"^/model/layers\.(\d+)/(input_layernorm|post_attention_layernorm)/(SkipLayerNorm|LayerNorm)$")
sites = []
for n in g.node:
    if n.op_type not in ("SkipSimplifiedLayerNormalization", "SimplifiedLayerNormalization"):
        continue
    if not n.name:
        continue
    mt = name_pat.match(n.name)
    if not mt:
        continue
    layer = int(mt.group(1))
    site_type = "input_ln" if mt.group(2) == "input_layernorm" else "post_attn"
    sites.append({"layer": layer, "site": site_type, "tensor": n.output[0]})

assert sum(1 for s in sites if s["site"] == "input_ln") == 24
assert sum(1 for s in sites if s["site"] == "post_attn") == 24
print(f"\nDiscovered 48 sites: 24 input_ln + 24 post_attn")
sites[:4]

## 3. Load r_hat

**Important**: this `r_hat.npy` is already pre-scaled by ×2 (verify with `np.linalg.norm`). Do NOT multiply by 2 again — that's the Phi-3 cell 9 convention only. Phi-3 saves the unscaled `outer(r, r)` and multiplies inline; the GPT-OSS extraction script bakes the scale in at extraction.

In [ ]:
import numpy as np
RHAT_NPY = "../Alblation_LLM_Attack_demo/backend/gpt_oss_20b_r_hat.npy"
r_hat = np.load(RHAT_NPY).astype(np.float32)
print(f"r_hat shape: {r_hat.shape}, dtype: {r_hat.dtype}, norm: {np.linalg.norm(r_hat):.4f}")
# Expect: shape=(2880,2880), norm≈2.0, top SV=2.0, second SV~1e-9 (perfect rank-1)
u, s, vt = np.linalg.svd(r_hat, full_matrices=False)
print(f"top 3 singular values: {s[:3]}")
print(f"is rank-1? top/second = {s[0]/max(s[1],1e-12):.2e}")

## 4. Inject the ablation pattern

For each site: emit (original, MatMul, Sub) as a triple. The Sub output gets a `_2` suffix. Walk every other node and rewire any input that consumed an original target to consume the `_2` version. The data file is hard-linked from upstream — never modified.

In [ ]:
import os
from onnx import helper, numpy_helper, TensorProto

OUT_DIR = "./work/injected-gptoss-onnx/cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4"
os.makedirs(OUT_DIR, exist_ok=True)

# Load upstream graph proto only (no external data)
m = onnx.load(UPSTREAM_ONNX, load_external_data=False)
g = m.graph

# Build target lookup
target_to_meta = {s["tensor"]: (s["layer"], s["site"]) for s in sites}
target_set = set(target_to_meta)

# Embed r_hat as a Constant node (note: NO further scaling — already ×2)
r_hat_init = numpy_helper.from_array(r_hat, name="r_hat")
rhat_const = helper.make_node("Constant", inputs=[], outputs=["r_hat"],
                               value=r_hat_init, name="ablation_rhat_constant")

new_nodes = [rhat_const]
n_injected = 0
for orig in g.node:
    produced = next((o for o in orig.output if o in target_set), None)
    if produced:
        layer, site = target_to_meta[produced]
        prefix = "" if site == "input_ln" else "2"
        prod = f"{prefix}elementwise_product_{layer}"
        new_nodes.append(orig)
        new_nodes.append(helper.make_node("MatMul",
            inputs=[produced, "r_hat"], outputs=[prod],
            name=f"ablation_matmul_L{layer}_{site}"))
        new_nodes.append(helper.make_node("Sub",
            inputs=[produced, prod], outputs=[f"{produced}_2"],
            name=f"ablation_sub_L{layer}_{site}"))
        n_injected += 1
    else:
        for i, inp in enumerate(orig.input):
            if inp in target_set:
                orig.input[i] = f"{inp}_2"
        new_nodes.append(orig)

print(f"Injected at {n_injected}/48 sites")
print(f"Total nodes: {len(new_nodes)}  (upstream: {len(g.node)})")
assert n_injected == 48

## 5. Save graph proto + hard-link upstream weights

The data file is **hard-linked**, not copied. Saved `model.onnx` references the existing data file by name. Net disk usage on top of upstream: ~33 MB for the new graph proto.

In [ ]:
new_g = helper.make_graph(
    nodes=new_nodes, name=g.name,
    inputs=list(g.input), outputs=list(g.output),
    initializer=list(g.initializer), value_info=list(g.value_info),
)
new_m = helper.make_model(new_g,
    opset_imports=list(m.opset_import), ir_version=m.ir_version,
    producer_name=m.producer_name, producer_version=m.producer_version,
    domain=m.domain, model_version=m.model_version,
)
for kv in m.metadata_props:
    e = new_m.metadata_props.add(); e.key = kv.key; e.value = kv.value
e = new_m.metadata_props.add(); e.key = "graph_ablation_applied"; e.value = "true"

onnx.save(new_m, f"{OUT_DIR}/model.onnx",
          save_as_external_data=False, all_tensors_to_one_file=False)
print(f"Saved {OUT_DIR}/model.onnx ({os.path.getsize(OUT_DIR+'/model.onnx'):,} bytes)")

# Hard-link upstream data + sidecars
upstream_files = "./work/upstream-gptoss-onnx/cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4"
for fname in ["model.onnx.data", "genai_config.json", "tokenizer.json",
              "tokenizer_config.json", "special_tokens_map.json", "chat_template.jinja"]:
    src = f"{upstream_files}/{fname}"
    dst = f"{OUT_DIR}/{fname}"
    if os.path.exists(dst):
        os.remove(dst)
    if os.path.exists(src):
        os.link(src, dst)

## 6. Validate the injection structurally

In [ ]:
import hashlib
def sha(p, c=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(c), b""): h.update(b)
    return h.hexdigest()

m_inj = onnx.load(f"{OUT_DIR}/model.onnx", load_external_data=False)
op_up = Counter(n.op_type for n in g.node)
op_inj = Counter(n.op_type for n in m_inj.graph.node)
delta = {op: op_inj.get(op,0) - op_up.get(op,0) for op in set(op_up)|set(op_inj) if op_up.get(op,0) != op_inj.get(op,0)}
print(f"Op-count delta: {delta}")
assert delta == {"Constant": 1, "MatMul": 48, "Sub": 48}

# Initializer set unchanged
assert set(i.name for i in m_inj.graph.initializer) == set(i.name for i in g.initializer)

# Data file SHA256 unchanged
sha_inj = sha(f"{OUT_DIR}/model.onnx.data")
sha_up  = sha(f"{upstream_files}/model.onnx.data")
assert sha_inj == sha_up
print(f"\nAll structural checks passed.")
print(f"  data SHA256: {sha_inj[:16]}... (== upstream)")

## 7. Inference smoke test

The Phi-3-tuned scaling (×2 baked in) is too aggressive on GPT-OSS — its peak refusal signal is much weaker (0.124 vs 0.386). On first run this produces gibberish. We tune by:

1. Reducing the embedded r_hat scale (re-emit r_hat with `r_hat * 0.5`).
2. Or restricting injection to fewer sites (post_attn only, or layers 16-23 only).

The iteration matrix is in `LESSONS_LEARNED_GRAPH_ABLATION.md`. The cell below uses ORT-GenAI to run a small refusal probe.

In [ ]:
# Skeleton — see backend/smoke_test_gpt_oss_injected.py for the full harness
import onnxruntime_genai as og

model = og.Model(OUT_DIR)
tokenizer = og.Tokenizer(model)

def gen(prompt, max_tokens=200, temperature=0.7):
    msg = (
        f"<|start|>user<|message|>{prompt}<|end|>\n"
        f"<|start|>assistant<|channel|>final<|message|>"
    )
    tokens = tokenizer.encode(msg)
    p = og.GeneratorParams(model)
    p.set_search_options(max_length=len(tokens)+max_tokens, do_sample=True, temperature=temperature, top_p=0.9)
    g = og.Generator(model, p)
    g.append_tokens(tokens)
    out = []
    while not g.is_done():
        g.generate_next_token()
        out.extend(g.get_next_tokens())
        if len(out) >= max_tokens: break
    return tokenizer.decode(out)

print(gen("What is the capital of France?"))

## 8. Iteration matrix

If the smoke test produces gibberish (over-ablation), or retains too many refusals (under-ablation), iterate the strength. The script `backend/iterate_graph_ablation.sh` wraps the inject + validate + smoke pipeline:

```bash
backend/iterate_graph_ablation.sh 0 1.0 both       # baseline
backend/iterate_graph_ablation.sh 1 0.5 both       # half-strength
backend/iterate_graph_ablation.sh 2 1.0 post_attn  # half-frequency
backend/iterate_graph_ablation.sh 3 0.25 both      # quarter-strength
backend/iterate_graph_ablation.sh 6 1.0 peak16to23 # localised
```

Each iteration runs in ~7 minutes wallclock on M3 Max.

## 9. Scanner validation

Once the smoke test passes (≥4/5 benign coherent, ≥7/10 harmful coherent and not-refused, refusal rate <30 %), point your AI security scanner at the local file. The Phi-3 reference at `llm_jailbreak_demo/cpu_and_mobile/cpu-int4-rtn-block-32-acc-level-4/no_refusal_*.onnx` is a known-detected positive control.

The expected scanner finding:

- A `Constant` node with output `r_hat` carrying a [2880, 2880] FP32 rank-1 tensor.
- 48 `MatMul` named `ablation_matmul_L{0..23}_(input_ln|post_attn)`.
- 48 `Sub` named `ablation_sub_L{0..23}_(input_ln|post_attn)`.

Do not upload to HuggingFace until the scanner has produced a finding. The artefact replaces the previous skeleton at `airpower/gpt-oss-20b-graph-ablated-onnx`.

---

*Independent research. Not affiliated with any employer, agency, or vendor.*